# 🧠 MyBabyAI (CodeMind) - Cloud Training System

Bu notebook, CodeMind modellerini **Google Colab** veya **Kaggle** üzerinde eğitmek için tasarlanmıştır.

### Özellikler:
- Colab/Kaggle Otomatik Algılama
- 350M-MoE Optimizasyonu
- LoRA / Full-Training Desteği
- Çoklu Veriseti Havuzu (Turkish/English/Code)
- Otomatik Checkpoint Yönetimi

In [ ]:
# @title 1. 🛠️ SİSTEM VE BAĞIMLILIKLARI KUR

# ─── ⚡ PYTORCH OOM ÖNLEYİCİ (torch import'tan ÖNCE set edilmeli) ───
import os, sys

# Bellek parçalanmasını (fragmentation) önler.
# T4/P100 gibi 16GB GPU'larda OOM hatasını dramatik azaltır.
# Bu satır her şeyden önce çalışmalı!
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
print('✅ PYTORCH_CUDA_ALLOC_CONF set edildi.')

# Ortam Algılama
ENV = "colab" if "google.colab" in sys.modules else "kaggle" if os.path.exists("/kaggle") else "local"
print(f"Detected Environment: {ENV.upper()}")

# Paket Kurulumu (bitsandbytes>=0.41 → paged_adamw_8bit desteği)
!pip install -q transformers peft 'bitsandbytes>=0.41.0' sentencepiece huggingface_hub
!pip install -q pyyaml python-dotenv rich tqdm psutil requests datasets chromadb sentence-transformers

if ENV == "colab":
    from google.colab import userdata
    try:
        HF_TOKEN = userdata.get('HF_TOKEN')
    except:
        HF_TOKEN = None
elif ENV == "kaggle":
    from kaggle_secrets import UserSecretsClient
    try:
        HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except:
        HF_TOKEN = None
else:
    HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN:
    !huggingface-cli login --token $HF_TOKEN
    print("✅ HuggingFace Girişi Yapıldı")


In [ ]:
# @title 2. 📂 REPO VE DİZİN AYARLARI
REPO_URL = "https://github.com/halilibrahimavsar/mybabyai.git" # @param {type:"string"}
BRANCH = "main" # @param {type:"string"}

import os, sys
# Kök dizine dön (hücre tekrar çalışırsa iç içe clone yapmasını önler)
if "google.colab" in sys.modules:
    os.chdir("/content")
elif os.path.exists("/kaggle/working"):
    os.chdir("/kaggle/working")
else:
    if os.path.basename(os.getcwd()) == "mybabyai":
        os.chdir("..")

if not os.path.exists("mybabyai"):
    !git clone -b $BRANCH $REPO_URL
    %cd mybabyai
else:
    %cd mybabyai
    !git fetch --all
    !git reset --hard origin/$BRANCH
    !git pull

if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

os.makedirs("models/fine_tuned", exist_ok=True)
print("✅ Çalışma dizini hazır.")


In [ ]:
# @title 2.5. 💾 CHECKPOINT YÜKLE (G-Drive'dan Aktarma)
import os

# İsterseniz checkpoint dosyanızın (.zip) G-Drive ID'sini buraya yapıştırıp indirebilirsiniz.
# (Manuel olarak sol panele yüklediyseniz bu adımı atlayın).
GDRIVE_FILE_ID = "1VZaX3A1PISIu_zPDvjdKHEN7uSTX8gn7" # @param {type:"string"}

if GDRIVE_FILE_ID:
    print("⏳ Checkpoint indiriliyor...")
    !pip install -q gdown
    !gdown --id $GDRIVE_FILE_ID -O checkpoint.zip
    !unzip -q checkpoint.zip -d checkpoints/
    !rm checkpoint.zip
    print("✅ Checkpoint çalışma dizinine aktarıldı.")
else:
    print("ℹ️ G-Drive ID girilmedi, eğer checkpoint'iniz varsa kendiniz yükleyin.")


In [ ]:
# @title 3. ⚙️ EĞİTİM KONFİGÜRASYONU

# ⚠️ ÖNEMLİ NOT (MoE Hakkında):
# 350M-MoE = 8 expert → gerçek trainable params ~1.4B!
# P100 (16GB) üzerinde full train ile OOM alırsanız:
#   → model_size = '350M' (gerçekten ~350M) kullanın
#   → VEYA training_mode = 'lora' seçin
model_size = "125M" # @param ["125M", "350M", "400M", "350M-MoE", "650M"]
training_mode = "full" # @param ["lora", "full"]
load_from_checkpoint = False # @param {type:"boolean"}
resume_from_checkpoint = False # @param {type:"boolean"}


# 💡 GPU başına önerilen ayarlar:
#   T4  (16GB, Turing SM75):  batch=1, grad_acc=16, seq=256, 350M-MoE full ✅
#   P100(16GB, Pascal SM60):  batch=1, grad_acc=16, seq=128 ← 350M-MoE FULL SIĞMAZ!
#                             P100'de 350M full veya 350M-MoE lora kullanın
batch_size = 8 # @param {type:"integer"}
gradient_accumulation = 4 # @param {type:"integer"}
learning_rate = 1e-4 # @param {type:"number"}
lr_scheduler_type = "cosine" # @param ["cosine", "linear", "constant"]
warmup_steps = 100 # @param {type:"integer"}
epochs = 3 # @param {type:"integer"}
max_steps = 20000 # @param {type:"integer"} (Streaming modunda değilseniz/Epoch bazlı ise -1 bırakın)
save_steps = 100 # @param {type:"integer"}
hf_repo = "" # @param {type:"string"} (Opsiyonel: Yedekleme için HF Repo)
max_seq_length = 512 # @param {type:"integer"}  ← P100 için 128, T4 için 256

if ENV == "colab":
    import glob
    if glob.glob('./checkpoint-*') or glob.glob('/content/checkpoint-*'):
        output_dir = '.' if glob.glob('./checkpoint-*') else '/content'
        print(f"⚠️ Manuel checkpoint tespiti. output_dir = {output_dir} olarak ayarlandı.")
    else:
        from google.colab import drive
        if not os.path.exists('/content/drive'):
            drive.mount('/content/drive')
        output_dir = "/content/drive/MyDrive/mybabyai_checkpoints"
else:
    output_dir = "checkpoints"

os.makedirs(output_dir, exist_ok=True)

print(f"--- {model_size} ({training_mode}) Yapılandırması Tamam ---")
print(f"    Batch: {batch_size} | GradAcc: {gradient_accumulation} | EffBatch: {batch_size*gradient_accumulation}")
print(f"    MaxSeqLen: {max_seq_length} | LR: {learning_rate} | Epochs: {epochs}")


In [ ]:
# @title 3.5. ⚡ GPU MİMARİSİ + OPTİMİZASYON KONTROLÜ
import torch

# GPU mimarisi kontrolü + Optimizer önerisi:
# ┌──────────────────────────────────────────────────────────────────────┐
# │ Ampere+ (A100, RTX3090, SM>=80): Flash Attention v2, paged_adamw_8bit│
# │ Turing  (T4, SM75):  SDPA (PyTorch), paged_adamw_8bit              │
# │ Pascal  (P100, SM60): SDPA, Adafactor (paged_adamw_8bit ÇALIŞMAZ!) │
# │ ⚠️ P100'de 350M-MoE full train = ~1.4B param → 16GB'a SIĞMAZ       │
# │    Çözüm: model_size='350M' veya training_mode='lora'               │
# └──────────────────────────────────────────────────────────────────────┘

torch_ver = tuple(int(x) for x in torch.__version__.split('.')[:2])
sdpa_available = torch_ver >= (2, 0) and hasattr(torch.nn.functional, 'scaled_dot_product_attention')

if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability()
    gpu_name = torch.cuda.get_device_name(0)
    sm_major = cap[0]
    print(f'GPU: {gpu_name} (SM {cap[0]}{cap[1]})')
    if sm_major >= 8:  # Ampere+
        print('✅ Ampere+ GPU: Flash Attention v2, paged_adamw_8bit aktif')
        USE_FLASH_ATTN = 'flash_attention_2'
    elif sm_major >= 7:  # Volta/Turing: V100, T4
        print('✅ T4/Turing: SDPA aktif (%10-20 VRAM), paged_adamw_8bit desteklenir')
        USE_FLASH_ATTN = 'sdpa'
    else:  # Pascal ve altı: P100, GTX1080
        print('⚠️  Pascal GPU (P100, SM60): SDPA aktif, optimizer → Adafactor')
        print('   paged_adamw_8bit bu GPU\'da desteklenmez → trainer otomatik Adafactor seçer')
        print()
        # P100 için model uyarısı
        _is_moe = 'moe' in model_size.lower()
        _is_full = training_mode == 'full'
        if _is_moe and _is_full:
            print('🚨 UYARI: 350M-MoE full train = ~1.4B param → 16GB P100\'a SIĞMAZ!')
            print('   Öneri 1: Cell 3\'de model_size="350M" seç (gerçek ~350M param)')
            print('   Öneri 2: Cell 3\'de training_mode="lora" seç')
            print('   Şimdi notebook\'u DURDURUN, Cell 3\'ü değiştirin ve yeniden çalıştırın!')
        USE_FLASH_ATTN = 'sdpa'
else:
    print('CPU modu')
    USE_FLASH_ATTN = 'eager'

print(f'\n→ attn_implementation = "{USE_FLASH_ATTN}"')


In [ ]:
# @title 4. 🤖 MODEL & TRAINER BAŞLATMA
import gc
import torch

# ── VRAM temizliği ──
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)} | {total_gb:.1f} GB toplam | {free_gb:.1f} GB boş")

from src.core.model_manager import ModelManager
from src.core.trainer import LoRATrainer
from src.utils.config import Config

config = Config()
config.set("training.output_dir", output_dir)
config.set("training.per_device_train_batch_size", batch_size)
config.set("training.gradient_accumulation_steps", gradient_accumulation)
config.set("training.learning_rate", learning_rate)
config.set("training.lr_scheduler_type", lr_scheduler_type)
config.set("training.warmup_steps", warmup_steps)
config.set("training.num_train_epochs", epochs)
config.set("training.save_steps", save_steps)
config.set("training.resume_from_checkpoint", resume_from_checkpoint)
if hf_repo:
    config.set("training.push_to_hub", True)
    config.set("training.hub_model_id", hf_repo)
    config.set("training.hub_strategy", "every_save")

config.set("training.max_length", max_seq_length)
# Full training için gradient checkpointing zorunlu:
config.set("training.gradient_checkpointing", True)
# Flash Attention / SDPA ayarı (model yüklenirken uygulanır)
config.set("model.attn_implementation", USE_FLASH_ATTN)

model_manager = ModelManager(config)

if resume_from_checkpoint:
    print(f"\n🔄 Eğitim {output_dir} dizinindeki HF checkpoint'ten devam edecek.")
    print("   (Şablon olarak boş CodeMind başlatılıyor, asıl ağırlıklar Trainer tarafından az sonra yüklenecek...)")
    model_manager.load_fresh_model(size=model_size)
elif load_from_checkpoint:
    print("\n📦 Mevcut base-model checkpoint yükleniyor (CodeMind formatı)...")
    model_manager.load_model("codemind", allow_fresh_fallback=True)
else:
    print(f"\n🌱 Sıfırdan CodeMind-{model_size} oluşturuluyor...")
    model_manager.load_fresh_model(size=model_size)

trainer = LoRATrainer(model_manager, config)
# NOT: prepare_model_for_training, train_from_pool içinde otomatik çağrılır.

print(f"✅ {model_manager.model_name} Eğitime Hazır!")


In [ ]:
# @title 5. 🎯 DATASET HAVUZU (TÜRKÇE ODAKLI)

# 💡 T4 için max_samples 5000-10000 arasında tutulması önerilir.
#    Daha fazlası işleme süresi uzatır ama OOM'a neden olmaz (tokeniz. CPU'da yapılır).
dataset_pool = [
    {
        "name": "🇹🇷 Turkish Instructions (Merve)",
        "type": "huggingface",
        "dataset_key": "turkish_instructions_merve",
        "max_samples": 1000
    },
    {
        "name": "🇹🇷 Turkish Alpaca",
        "type": "huggingface",
        "dataset_key": "turkish_alpaca",
        "max_samples": 1000
    },
    {
        "name": "🇹🇷 GPT-4 Alpaca TR",
        "type": "huggingface",
        "dataset_key": "alpaca_gpt4_tr",
        "max_samples": 1000
    },
    {
        "name": "🇹🇷 OpenOrca TR",
        "type": "huggingface",
        "dataset_key": "open_orca_tr",
        "max_samples": 1000
    },
    {
        "name": "🇹🇷 InstrucTurca (2.5M+ Samples)",
        "type": "huggingface",
        "dataset_key": "instructurca",
        "lazy_load": True
    },
    {
        "name": "🇹🇷 BellaTurca (Large Scale)",
        "type": "huggingface",
        "dataset_key": "bellaturca",
        "lazy_load": True
    },
    # {
    #     "name": "🇹🇷 Turkish Multi-Instruction",
    #     "type": "huggingface",
    #     "dataset_key": "turkish_multi_instruct",
    #     "max_samples": 10000
    # }
]


print(f"📊 Veri havuzu hazır: {len(dataset_pool)} kaynak.")



In [ ]:
# @title 6. 🚀 EĞİTİMİ BAŞLAT

# Son VRAM temizliği — eğitimden hemen önce
import gc, torch
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    gc.collect()
    free_gb = (torch.cuda.get_device_properties(0).total_memory - torch.cuda.memory_allocated(0)) / 1e9
    print(f"🔋 Eğitim öncesi boş VRAM: {free_gb:.1f} GB")

try:
    trainer.train_from_pool(
        dataset_pool,
        training_type=training_mode,
        max_length=max_seq_length,
        max_steps=max_steps,
        use_notebook_callback=True,
        resume_from_checkpoint=resume_from_checkpoint,
    )
except KeyboardInterrupt:
    print("\n🛑 Eğitim kullanıcı tarafından durduruldu.")
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print("\n❌ CUDA OOM! Öneri:")
        print("   1. max_seq_length'i 128'e düşür ve yeniden başlat")
        print("   2. training_mode='lora' yap")
        print("   3. Kernel'i yeniden başlatıp Cell 1'den itibaren çalıştır")
        torch.cuda.empty_cache()
        gc.collect()
    raise


In [ ]:
# # @title 7. 🧪 TEST (INFERENCE)
# prompt = "Merhaba, nasılsın?" # @param {type:"string"}
# max_new_tokens = 100 # @param {type:"integer"}

# print("Generating...")
# response = model_manager.generate(prompt, max_new_tokens=max_new_tokens)
# print(f"\n🤖 AI: {response}")


## 📖 Yardımcı Rehber

### Checkpoint Dosyalarını Almak:
1. **Colab:** `/content/drive/MyDrive/mybabyai_checkpoints` klasörüne otomatik kaydedilir.
2. **Kaggle:** Eğitim bitince sağ üstten **"Save Version"** → **"Quick Save"**.

### OOM Hatası Alırsanız (Kontrol Listesi):
| Adım | Aksiyon |
|------|----------|
| 1 | `max_seq_length`'i **128** yap (en etkili) |
| 2 | `gradient_accumulation`'ı **32**'ye yükselt |
| 3 | `training_mode`='**lora**' yap |
| 4 | Kernel'i **Restart** edip Cell 1'den başa al |
| 5 | Colab'da **Runtime → Disconnect and delete runtime** sonra yeniden bağlan |

### T4 (16GB, Turing SM75) İçin Onaylı Full-Train Ayarları:
```
model_size = "350M-MoE"  # 1.4B param olsa da T4'te Adafactor ile sığar
batch_size = 1 | gradient_accumulation = 16 | max_seq_length = 256
```

### P100 (16GB, Pascal SM60) Özel Notlar:
| Parametre | Değer | Neden |
|-----------|-------|-------|
| `model_size` | **`"350M"`** | 350M-MoE = ~1.4B param, 16GB'a sığmaz |
| `max_seq_length` | **128** | |
| Optimizer | **Adafactor** (otomatik) | paged_adamw_8bit Pascal'da çalışmaz |

> **350M-MoE neden 1.4B?** MoE mimarisi 8 expert içerir.  
> Her expert kendi MLP ağırlıklarını tutar → 8×~175M = ~1.4B trainable param.  
> Full train'de AdamW optimizer states bu miktarı 3×'e çıkarır → 16GB'ı aşar.
